## Imports

In [3]:
from IPython import display
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import sys
import os
import pandas as pd
import numpy as np


path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if path not in sys.path:
    sys.path.append(path)
    
from database.database import engine, SessionLocal
from models.sensor_data import Base, SensorData
from services.sensor_service import carregar_dados_bd




## Carregando os dados do banco de dados

In [4]:


df = carregar_dados_bd()

print(df.shape)
df.head()


Carregando dados do Banco de Dados...
Dados carregados com sucesso!
(166796, 27)


,_sa_instance_state,temperature_f,z_rms_acceleration_g,x_peak_velocity_in_s,temperature_c,x_rms_acceleration_g,x_peak_velocity_mm_s,x_rms_velocity_in_s,z_kurtosis,z_high_freq_rms_accel_g,...,id,x_peak_acceleration_g,x_crest_factor,z_rms_velocity_in_s,z_peak_vel_comp_freq_hz,z_peak_velocity_in_s,rpm,z_rms_velocity_mm_s,x_peak_vel_comp_freq_hz,z_peak_velocity_mm_s
0,<sqlalchemy.orm.state.InstanceState object at ...,74.00,0.046,0.0875,23.33,0.066,2.224,0.0619,3.276,0.007,...,1426,0.033,3.918,0.0427,61.0,0.0605,0.0,1.086,61.0,1.536
1,<sqlalchemy.orm.state.InstanceState object at ...,73.91,0.046,0.0880,23.28,0.067,2.236,0.0622,3.447,0.007,...,1427,0.040,4.924,0.0431,61.0,0.0609,0.0,1.095,61.0,1.549
2,<sqlalchemy.orm.state.InstanceState object at ...,74.00,0.046,0.0880,23.33,0.067,2.236,0.0622,3.447,0.007,...,1428,0.040,4.924,0.0431,61.0,0.0609,0.0,1.095,61.0,1.549
3,<sqlalchemy.orm.state.InstanceState object at ...,74.00,0.046,0.0873,23.33,0.066,2.218,0.0617,7.921,0.007,...,1429,0.100,10.929,0.0431,61.0,0.0610,0.0,1.096,61.0,1.550
4,<sqlalchemy.orm.state.InstanceState object at ...,73.96,0.047,0.0870,23.31,0.066,2.210,0.0615,3.245,0.007,...,1430,0.049,5.949,0.0440,61.0,0.0623,0.0,1.119,61.0,1.583


Verificação das infos

In [5]:
print(df.shape)
print(df.head())
print(df.info())

(166796, 27)
                                  _sa_instance_state  temperature_f  \
0  <sqlalchemy.orm.state.InstanceState object at ...          74.00   
1  <sqlalchemy.orm.state.InstanceState object at ...          73.91   
2  <sqlalchemy.orm.state.InstanceState object at ...          74.00   
3  <sqlalchemy.orm.state.InstanceState object at ...          74.00   
4  <sqlalchemy.orm.state.InstanceState object at ...          73.96   

   z_rms_acceleration_g  x_peak_velocity_in_s  temperature_c  \
0                 0.046                0.0875          23.33   
1                 0.046                0.0880          23.28   
2                 0.046                0.0880          23.33   
3                 0.046                0.0873          23.33   
4                 0.047                0.0870          23.31   

   x_rms_acceleration_g  x_peak_velocity_mm_s  x_rms_velocity_in_s  \
0                 0.066                 2.224               0.0619   
1                 0.067            

verificando os nulos

In [6]:
df.isnull().sum()

_sa_instance_state         0
temperature_f              0
z_rms_acceleration_g       0
x_peak_velocity_in_s       0
temperature_c              0
x_rms_acceleration_g       0
x_peak_velocity_mm_s       0
x_rms_velocity_in_s        0
z_kurtosis                 0
z_high_freq_rms_accel_g    0
x_rms_velocity_mm_s        0
x_kurtosis                 0
x_high_freq_rms_accel_g    0
z_peak_acceleration_g      0
z_crest_factor             0
fault                      0
created_at                 0
id                         0
x_peak_acceleration_g      0
x_crest_factor             0
z_rms_velocity_in_s        0
z_peak_vel_comp_freq_hz    0
z_peak_velocity_in_s       0
rpm                        0
z_rms_velocity_mm_s        0
x_peak_vel_comp_freq_hz    0
z_peak_velocity_mm_s       0
dtype: int64

Distribuição das falhas

In [7]:
df["fault"].value_counts()
print(df["fault"].value_counts(normalize=True))

fault
rolamento_inner            0.077940
eccentric_rotor            0.070793
desbalanceado_1parafuso    0.060427
cocked_rotor               0.059953
rolamento_outer            0.059953
                             ...   
dedesbalanceado_adxl_1     0.000126
normal_6                   0.000120
acelerando                 0.000042
new_tes                    0.000012
new_teste                  0.000012
Name: proportion, Length: 151, dtype: float64


Removendo colunas que não auxiliam

In [8]:
df = df.drop(columns=[
    "_sa_instance_state",
    "id",
    "created_at"
])

Vou diminuir a amostra para treino (pc mais fraco)

In [9]:
df_amostra = df.sample(frac=0.8, random_state=42)

# Amostragem para evitar travamentos (só trocar na var acima a porcentagem, deixei 80%)
df = df_amostra
print(df.shape)

(133437, 24)


Setando X e Y

In [10]:
X = df.drop("fault", axis=1)

y = df["fault"]

In [11]:
# Tratando pouca amostra e nome sujo
min_amostras = 30
contagem = df["fault"].value_counts()
classes_boas = contagem[contagem >= min_amostras].index
df = df[df["fault"].isin(classes_boas)]
print("classes depois do filtro:", df["fault"].nunique())
print(df.shape)

classes depois do filtro: 145
(133368, 24)


separar features e target

In [12]:
X = df.drop("fault", axis=1)
y = df["fault"]
print(X.shape)
print(y.shape)

(133368, 23)
(133368,)


Separando o teste e treino

In [13]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
#Func para avaliar modelos
def avaliar(modelo, nome):

            print(f"Treinando {nome}...")

            modelo.fit(X_train, y_train)

            pred = modelo.predict(X_test)

            print(classification_report(y_test, pred))

            resultado = {
                "Modelo": nome,
                "Accuracy": accuracy_score(y_test, pred),
                "Precision": precision_score(y_test, pred, average="weighted"),
                "Recall": recall_score(y_test, pred, average="weighted"),
                "F1": f1_score(y_test, pred, average="weighted")
            }

            print(resultado)

            return resultado

Para guardar os resultados

In [15]:
resultados = []

Random forest

In [16]:
modelo = RandomForestClassifier(
            n_estimators=50,
            max_depth=10,
            random_state=42,
            n_jobs=2
        ) #limitei aqui por conta do pc usado nos testes

resultados.append(
    avaliar(modelo, "Random Forest")
)

Treinando Random Forest...
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         8
           1       1.00      0.50      0.67        16
           2       0.47      0.60      0.53      1602
           3       0.78      0.52      0.62       483
           4       0.95      0.79      0.86        24
           5       1.00      0.62      0.77        24
           6       0.65      0.54      0.59        24
           7       0.64      0.90      0.75      1433
           8       0.72      0.55      0.63       472
           9       0.75      0.75      0.75         8
          10       1.00      0.25      0.40         8
          11       0.46      0.88      0.61       644
          12       0.55      0.50      0.52       633
          13       0.00      0.00      0.00        24
          14       0.75      0.72      0.73      1611
          15       0.88      0.19      0.31        37
          16       0.80      0.50      0.62        24


d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

Extra trees

In [17]:
modelo = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=2
)

resultados.append(
    avaliar(modelo, "Extra Trees")
)

Treinando Extra Trees...
              precision    recall  f1-score   support

           0       1.00      0.88      0.93         8
           1       0.93      0.88      0.90        16
           2       0.78      0.80      0.79      1602
           3       0.82      0.86      0.84       483
           4       0.96      0.96      0.96        24
           5       1.00      0.92      0.96        24
           6       0.95      0.88      0.91        24
           7       0.89      0.96      0.92      1433
           8       0.87      0.79      0.83       472
           9       0.89      1.00      0.94         8
          10       1.00      0.62      0.77         8
          11       0.77      0.88      0.82       644
          12       0.76      0.81      0.78       633
          13       1.00      0.25      0.40        24
          14       0.85      0.88      0.87      1611
          15       0.81      0.70      0.75        37
          16       0.92      0.92      0.92        24
  

HGB

In [18]:
from sklearn.ensemble import HistGradientBoostingClassifier

modelo = HistGradientBoostingClassifier(max_iter=50, max_depth=10, 
    random_state=42
)

resultados.append(
    avaliar(modelo, "HistGradientBoosting")
)

Treinando HistGradientBoosting...
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         8
           1       0.24      0.56      0.34        16
           2       0.61      0.56      0.59      1602
           3       0.74      0.62      0.68       483
           4       0.00      0.00      0.00        24
           5       0.02      0.04      0.02        24
           6       0.03      0.12      0.04        24
           7       0.80      0.77      0.78      1433
           8       0.58      0.46      0.51       472
           9       0.00      0.00      0.00         8
          10       0.10      0.38      0.15         8
          11       0.63      0.63      0.63       644
          12       0.62      0.57      0.59       633
          13       0.04      0.08      0.06        24
          14       0.65      0.47      0.54      1611
          15       0.25      0.43      0.32        37
          16       0.10      0.42      0.17    

d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

XGBoost

In [ ]:
from xgboost import XGBClassifier

modelo = XGBClassifier(n_estimators=50, max_depth=40,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist"
)

resultados.append(
    avaliar(modelo, "XGBoost")
)

Treinando XGBoost...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      0.94      0.97        16
           2       0.79      0.79      0.79      1602
           3       0.80      0.83      0.82       483
           4       0.89      1.00      0.94        24
           5       0.95      0.88      0.91        24
           6       0.88      0.88      0.88        24
           7       0.92      0.96      0.94      1433
           8       0.85      0.81      0.83       472
           9       1.00      0.88      0.93         8
          10       1.00      0.62      0.77         8
          11       0.78      0.83      0.80       644
          12       0.72      0.74      0.73       633
          13       0.70      0.29      0.41        24
          14       0.86      0.88      0.87      1611
          15       0.67      0.76      0.71        37
          16       0.77      0.83      0.80        24
      

LightGBM

In [20]:
from lightgbm import LGBMClassifier

modelo = LGBMClassifier(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.1,
    num_class=len(np.unique(y_train)),
    objective="multiclass",
    random_state=42,
    verbose=-1
)

resultados.append(
    avaliar(modelo, "LightGBM")
)

Treinando LightGBM...
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         8
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00      1602
           3       0.00      0.00      0.00       483
           4       0.00      0.00      0.00        24
           5       0.00      0.00      0.00        24
           6       0.00      0.00      0.00        24
           7       0.00      0.00      0.00      1433
           8       0.00      0.00      0.00       472
           9       0.00      0.00      0.00         8
          10       0.00      0.00      0.00         8
          11       0.09      0.02      0.03       644
          12       0.43      0.01      0.02       633
          13       0.00      0.00      0.00        24
          14       0.41      0.08      0.14      1611
          15       0.00      0.00      0.00        37
          16       0.00      0.00      0.00        24
     

d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projetos\Py\Fullstack_ia_python\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

CatBoost

In [21]:
from catboost import CatBoostClassifier

modelo = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    random_state=42,
    verbose=100
)

resultados.append(
    avaliar(modelo, "CatBoost")
)

Treinando CatBoost...
0:	learn: 3.7904056	total: 2.62s	remaining: 13m 2s
100:	learn: 1.0326513	total: 3m 43s	remaining: 7m 20s
200:	learn: 0.7759786	total: 7m 24s	remaining: 3m 39s
299:	learn: 0.6543408	total: 11m 9s	remaining: 0us
              precision    recall  f1-score   support

           0       0.78      0.88      0.82         8
           1       1.00      0.94      0.97        16
           2       0.69      0.70      0.70      1602
           3       0.72      0.77      0.74       483
           4       0.88      0.96      0.92        24
           5       1.00      0.92      0.96        24
           6       1.00      0.83      0.91        24
           7       0.87      0.94      0.90      1433
           8       0.79      0.68      0.73       472
           9       1.00      0.75      0.86         8
          10       0.60      0.38      0.46         8
          11       0.67      0.81      0.73       644
          12       0.61      0.69      0.65       633
          1

### Ranking

In [22]:
ranking = pd.DataFrame(resultados)

ranking.sort_values(
    "F1",
    ascending=False
)

,Modelo,Accuracy,Precision,Recall,F1
1,Extra Trees,0.852328,0.853604,0.852328,0.851419
3,XGBoost,0.850791,0.851377,0.850791,0.850317
5,CatBoost,0.776899,0.777607,0.776899,0.775533
0,Random Forest,0.635450,0.670736,0.635450,0.629845
2,HistGradientBoosting,0.501087,0.598927,0.501087,0.534876
4,LightGBM,0.071380,0.095302,0.071380,0.029904


### Busca de Hiperparâmetros (RandomizedSearchCV)

O XGBoost teve o melhor resultado no ranking acima, então faço aqui a busca dos melhores hiperparâmetros para ele (essa busca foi movida do `TrainService`, que agora usa os parâmetros já encontrados).

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

parametros = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [4, 6, 8, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
} # treina X(100,200,300) modelos na memória, com X(4,6,8,10) de profundidade e X(0.8,1.0) de subsample e colsample_bytree

modelo = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist"
)

busca = RandomizedSearchCV(
    estimator=modelo,
    param_distributions=parametros,
    n_iter=20,
    cv=5,
    scoring="f1_weighted",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

busca.fit(X_train, y_train)

print("\nMelhores parâmetros:")
print(busca.best_params_)

Os melhores hiperparâmetros foram aplicado no train_service (que no fim não deu uma alteração muito grande de uma aplicação mais simples que eu tinha feito no commit anterior)